In [ ]:
import pandas as pd
import os

INPUT_PATH = "../../data/raw/combined_2024_2025.csv"
OUTPUT_PATH = "../../data/processed/jisoo_combined_2024_2025_processed.csv"

REGIONS = [
    "서울", "부산", "대구", "인천", "광주", "대전", "울산", "세종", "경기",
    "강원", "충북", "충남", "전북", "전남", "경북", "경남", "제주",
]
VALID_CASE_CODES = {1, 2, 3, 4, 5}
TRA_PREFIXES = [f"D_TRA{n}_" for n in range(1, 7)]
TRA_SUFFIXES = ["SYEAR", "SMONTH", "1_SPOT", "CASE", "COST", "NUM", "ONE_COST"]
REGION_COUNT_COLS = [f"국내_A_여행횟수_관광전체_{r}" for r in REGIONS]
REGION_COST_COLS = [f"국내_A_여행지출_관광전체_{r}" for r in REGIONS]

df = pd.read_csv(INPUT_PATH)
print("데이터 로드:", df.shape)

# STEP 2. 타입 변환 (float to int)
# D_TRA1~6 관련 42개 컬럼은 회차별 구조적 결측 때문에 float64로 저장되어 있음.
# 값 자체는 float이어도 로직상 문제없지만, CSV/태블로에서 볼 때 지저분하므로 정리.
# 결측이 있어 일반 int64는 못쓰고 Int64(Nullable Integer) 사용.
tra_cols = [f"{p}{s}" for p in TRA_PREFIXES for s in TRA_SUFFIXES if f"{p}{s}" in df.columns]
for col in tra_cols:
    df[col] = df[col].astype("Int64")
if "연도" in df.columns:
    df["연도"] = df["연도"].astype("int64")
print("STEP2 타입 변환 완료, 대상 컬럼 수:", len(tra_cols))

# STEP 3. 결측치 확인 (대체하지 않고 원본 유지)
# D_TRAn_* 결측은 응답 누락이 아니라 n번째 여행을 하지 않았다는 구조적 결측이라
# 0이나 평균으로 채우면 존재하지 않는 사실을 만들어내므로 대체하지 않음.
rows = []
for n in range(1, 7):
    rep_col = f"D_TRA{n}_SYEAR"
    if rep_col in df.columns:
        rate = df[rep_col].isna().mean() * 100
        rows.append({"회차": f"{n}차", "결측률": round(rate, 1)})
print("STEP3 결측률 확인")
print(pd.DataFrame(rows))

# STEP 5. 이상치 판정 (원본 값은 수정하지 않음)
# 전체 응답자 기준 Winsorize는 비방문자(지출 0)가 대부분인 영과잉 분포를 왜곡시키므로
# 방문자 집합에서만 IQR x3(Outer Fence) 기준으로 이상치를 판정함.
# 지역별 17개 플래그 대신 이상치여부/이상치지역 2개 컬럼으로 압축.
region_outlier_masks = {}
for region in REGIONS:
    count_col = f"국내_A_여행횟수_관광전체_{region}"
    cost_col = f"국내_A_여행지출_관광전체_{region}"
    visitors = df[count_col] > 0
    visitor_costs = df.loc[visitors, cost_col]
    q1, q3 = visitor_costs.quantile([0.25, 0.75])
    iqr = q3 - q1
    upper_fence = q3 + 3 * iqr
    lower_fence = q1 - 3 * iqr
    outlier_mask = visitors & ((df[cost_col] > upper_fence) | (df[cost_col] < lower_fence))
    region_outlier_masks[region] = outlier_mask

outlier_df = pd.DataFrame(region_outlier_masks, index=df.index)
df["파생_이상치여부"] = outlier_df.any(axis=1).astype(int)
df["파생_이상치지역"] = outlier_df.apply(lambda row: ",".join(row.index[row]), axis=1)
print("STEP5 이상치 판정 완료, 보유자 수:", int(df["파생_이상치여부"].sum()))

# STEP 6. 데이터 유효성 통합 검증 및 손상 레코드 제거
# 그룹A(D_TRA1 필드 자체 유효성)와 그룹B(레코드 전체 논리 정합성)는
# 값 자체가 이상한지 검증한다는 동일 성격이라 하나로 통합함.
violation_mask = pd.Series(False, index=df.index)
results = []

has_tra1 = df["D_TRA1_SYEAR"].notna()
tra1 = df.loc[has_tra1]
expected_one_cost = tra1["D_TRA1_COST"] / tra1["D_TRA1_NUM"]

group_a = {
    "A1_SYEAR범위": ~tra1["D_TRA1_SYEAR"].isin([2024, 2025]),
    "A2_SMONTH범위": ~tra1["D_TRA1_SMONTH"].between(1, 12),
    "A3_CASE유효성": ~tra1["D_TRA1_CASE"].isin(VALID_CASE_CODES),
    "A4_COST음수": tra1["D_TRA1_COST"] < 0,
    "A5_NUM1명미만": tra1["D_TRA1_NUM"] < 1,
    "A6_ONECOST정합성": (expected_one_cost - tra1["D_TRA1_ONE_COST"]).abs() > 1,
}
for name, mask in group_a.items():
    results.append({"항목": name, "위반건수": int(mask.sum())})
    violation_mask.loc[tra1.index] |= mask.reindex(tra1.index, fill_value=False)

b2 = pd.Series(False, index=df.index)
for region in REGIONS:
    count_col = f"국내_A_여행횟수_관광전체_{region}"
    cost_col = f"국내_A_여행지출_관광전체_{region}"
    b2 |= (df[count_col] == 0) & (df[cost_col] > 0)

group_b = {
    "B1_WTDOM이상": df["WT_DOM"].isna() | (df["WT_DOM"] <= 0),
    "B2_지출횟수불일치": b2,
    "B3_연도범위외": ~df["연도"].isin([2024, 2025]),
}
for name, mask in group_b.items():
    results.append({"항목": name, "위반건수": int(mask.sum())})
    violation_mask |= mask

print("STEP6 유효성 검증 결과")
print(pd.DataFrame(results))

# 그룹C 참고 진단, 손상 아님. 지역합계는 관광 목적만 집계하므로
# 단순출장만 다녀온 사람은 지역합계에서 0으로 빠짐.
case_cols = [c for c in df.columns if c.endswith("_CASE")]
trip_count = df[case_cols].notna().sum(axis=1)
region_trip_count = df[REGION_COUNT_COLS].sum(axis=1)
mismatch = (trip_count != region_trip_count).sum()
print("STEP6C 전체목적여행횟수와 관광목적지역합계 불일치 인원 수:", mismatch)

n_before = len(df)
df = df.loc[~violation_mask].copy()
print("STEP6 위반 레코드 제거:", n_before, "건에서", len(df), "건으로")

# STEP 7. 파생변수 생성
# 17개 지역별로 쪼개진 값만으로는 응답자 단위 요약이 어려워 지표 5개를 만듦.
df["파생_총방문지역수"] = (df[REGION_COUNT_COLS] > 0).sum(axis=1)
df["파생_총여행횟수"] = df[REGION_COUNT_COLS].sum(axis=1)
df["파생_총여행지출"] = df[REGION_COST_COLS].sum(axis=1)
df["파생_여행경험여부"] = (df["파생_총여행횟수"] > 0).astype(int)
df["파생_1회평균지출"] = df["파생_총여행지출"] / df["파생_총여행횟수"]
df.loc[df["파생_총여행횟수"] == 0, "파생_1회평균지출"] = pd.NA
print("STEP7 파생변수 5개 생성 완료")
print("최종 shape:", df.shape)

# 저장
os.makedirs("../../data/processed", exist_ok=True)
df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8-sig")
print("저장 완료:", OUTPUT_PATH)

데이터 로드: (103939, 82)
STEP2 타입 변환 완료, 대상 컬럼 수: 42
STEP3 결측률 확인
   회차    결측률
0  1차   49.0
1  2차   97.1
2  3차   99.8
3  4차  100.0
4  5차  100.0
5  6차  100.0
STEP5 이상치 판정 완료, 보유자 수: 699
STEP6 유효성 검증 결과
              항목  위반건수
0     A1_SYEAR범위     0
1    A2_SMONTH범위     0
2     A3_CASE유효성     0
3      A4_COST음수     0
4     A5_NUM1명미만     0
5  A6_ONECOST정합성     0
6     B1_WTDOM이상     0
7     B2_지출횟수불일치     0
8       B3_연도범위외     0
STEP6C 전체목적여행횟수와 관광목적지역합계 불일치 인원 수: 7496
STEP6 위반 레코드 제거: 103939 건에서 103939 건으로
STEP7 파생변수 5개 생성 완료
최종 shape: (103939, 89)
저장 완료: ../../data/processed/preprocessed_combined.csv
